In [1]:
import os
import pandas as pd
import re
import numpy as np
import warnings
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
DATA_DIR = Path("../data")

files = sorted(DATA_DIR.glob("*.tsv"))

for f in files:
    print(f.name)

swan_cross_sectional.tsv
swan_visit_0.tsv
swan_visit_1.tsv
swan_visit_10.tsv
swan_visit_2.tsv
swan_visit_3.tsv
swan_visit_4.tsv
swan_visit_5.tsv
swan_visit_6.tsv
swan_visit_7.tsv
swan_visit_8.tsv
swan_visit_9.tsv


In [3]:
visit_files = sorted(
    [f for f in files if "visit" in f.name.lower()],
    key=lambda x: int(re.search(r'visit_(\d+)', x.name).group(1))
)

for f in visit_files:
    print(f.name)

swan_visit_0.tsv
swan_visit_1.tsv
swan_visit_2.tsv
swan_visit_3.tsv
swan_visit_4.tsv
swan_visit_5.tsv
swan_visit_6.tsv
swan_visit_7.tsv
swan_visit_8.tsv
swan_visit_9.tsv
swan_visit_10.tsv


In [4]:
check_vars = [
    "DHAS",
    "SYSBP1",
    "DIABP1",
    "WAIST",
    "HIP",
    "HDL",
    "TRIG",
    "INSUR",
    "CRP",
    "TRBLSLE",
    "TYPNIGH",
    "SLEEPQL"
]

for f in visit_files:
    df = pd.read_csv(f, sep="\t", low_memory=False)

    print(f"\n{f.name}")

    for stem in check_vars:
        hits = [c for c in df.columns if stem in c]
        print(stem, "->", hits[:5])


swan_visit_0.tsv
DHAS -> ['DHAS0']
SYSBP1 -> ['SYSBP10']
DIABP1 -> ['DIABP10']
WAIST -> ['WAIST0']
HIP -> ['HIPBRK0', 'HIPAGE0', 'HIP0', 'HIPMEAS0']
HDL -> ['HDLRESU0']
TRIG -> ['TRIGRES0']
INSUR -> ['NOINSUR0', 'INSURES0']
CRP -> ['CRPRESU0']
TRBLSLE -> ['TRBLSLE0']
TYPNIGH -> ['TYPNIGH0']
SLEEPQL -> []

swan_visit_1.tsv
DHAS -> ['DHAS1']
SYSBP1 -> ['SYSBP11']
DIABP1 -> ['DIABP11']
WAIST -> ['WAIST1']
HIP -> ['HIP1', 'HIPMEAS1']
HDL -> ['HDLRESU1']
TRIG -> ['TRIGRES1']
INSUR -> ['INSURES1']
CRP -> ['CRPRESU1']
TRBLSLE -> ['TRBLSLE1']
TYPNIGH -> ['TYPNIGH1']
SLEEPQL -> []

swan_visit_2.tsv
DHAS -> ['DHAS2']
SYSBP1 -> ['SYSBP12']
DIABP1 -> ['DIABP12']
WAIST -> ['WAIST2']
HIP -> ['HIP2', 'HIPMEAS2']
HDL -> []
TRIG -> []
INSUR -> []
CRP -> []
TRBLSLE -> ['TRBLSLE2']
TYPNIGH -> ['TYPNIGH2']
SLEEPQL -> []

swan_visit_3.tsv
DHAS -> ['DHAS3']
SYSBP1 -> ['SYSBP13']
DIABP1 -> ['DIABP13']
WAIST -> ['WAIST3']
HIP -> ['HIP3', 'HIPMEAS3']
HDL -> ['HDLRESU3']
TRIG -> ['TRIGRES3']
INSUR -> ['INSURES

# Join dataset

In [5]:
all_visits = []

for f in visit_files:
    df_temp = pd.read_csv(f, sep="\t", low_memory=False)
    
    # Extract the visit number from the filename (e.g., '0' from 'swan_visit_0.tsv')
    match = re.search(r'visit_(\d+)', f.name)
    if match:
        visit_num = match.group(1)
        
        # 3. Rename columns: remove the visit number suffix from the end of the string
        # This handles the case where visit 10 is '10' and visit 1 is '1'
        # e.g., 'DIABP110' (Visit 10) -> 'DIABP1'
        # e.g., 'DIABP11' (Visit 1) -> 'DIABP1'
        df_temp.columns = [re.sub(rf'{visit_num}$', '', col) if col not in ['SWANID', 'VISIT'] else col 
                          for col in df_temp.columns]
        
        # 4. Ensure the VISIT column is consistent (some files use 'VISIT', others might vary)
        if 'VISIT' not in df_temp.columns:
            df_temp['VISIT'] = int(visit_num)
            
        all_visits.append(df_temp)

# 5. Merge all visits into one long dataset
df_long = pd.concat(all_visits, axis=0, ignore_index=True)

# Inspect the result
print(f"Combined Shape: {df_long.shape}")
print(df_long[['SWANID', 'VISIT', 'DIABP1', 'SYSBP1']].head())

Combined Shape: (28789, 2102)
   SWANID  VISIT DIABP1 SYSBP1
0   10005      0     80    114
1   10046      0     58    120
2   10056      0     60     92
3   10092      0     70    108
4   10126      0     72     98


In [6]:
missing_codes = [-9, "-9", -8, "-8", -7, "-7", -1, "-1", ' ', '.', ''] #Some string variables were sturviving after the filter, so remove those

df_long = df_long.replace(missing_codes, np.nan)

In [7]:
#Check random participant format
participant_10245 = df_long[df_long['SWANID'] == 10245]

print(participant_10245)

       SWANID  VISIT INTDAY AGE PREGNAN PREVBLO ALCHL24 EATDRIN STRTPER  \
7       10245      0      0  47       1     NaN       1       1       2   
3308    10245      1    355  48       1     NaN       1       1       2   
6188    10245      2    661  49       1       1       1       1       1   
8936    10245      3   1032  50       1       1       1       1       1   
11645   10245      4   1452  51       1       1       2       1       1   
14324   10245      5   1797  52       1       1       1       1       1   
16941   10245      6   2296  54       1       1       1       1       1   
19388   10245      7   2511  54       1     NaN       2       1       1   
21802   10245      8   2923  55     NaN     NaN     NaN     NaN     NaN   
24080   10245      9   3265  56       1     NaN       2       1       1   
26548   10245     10   3612  57       1     NaN       1       1       1   

      BLDRWAT  ... MEALBAR  INSTSHK  CHOCOCD  ICECREA  SALADDR  DRKBEER  \
7           1  ...     N

In [13]:
key_vars = [
    "DHAS",
    "SYSBP1","SYSBP2",
    "DIABP1","DIABP2",
    "WAIST","HIP",
    "HDLRESU",
    "TRIGRES",
    "INSURES",
    "CRPRESU",
    "TYPNIGH",
    "GLUCRES",
    "SLEEPQL"
]

df_long[key_vars].isna().mean().sort_values() ##Check missingness

SYSBP1     0.098822
DIABP1     0.099239
SYSBP2     0.099830
DIABP2     0.100108
WAIST      0.104832
HIP        0.105526
DHAS       0.137344
SLEEPQL    0.389663
HDLRESU    0.412380
CRPRESU    0.418597
GLUCRES    0.436139
TRIGRES    0.437285
INSURES    0.447706
TYPNIGH    0.602904
dtype: float64

In [14]:
numeric_cols = ["DHAS", "SYSBP1", "SYSBP2", "SYSBP3", "DIABP1", "DIABP2", "DIABP3",
                "WAIST", "HIP", "HDLRESU", "TRIGRES", "INSURES", "CRPRESU", "TYPNIGH", "SLEEPQL", "LDLRESU", "CHOLRES", "GLUCRES", "BMI", "TRBLSLE", "WAKEUP", "WAKEARL"]

for col in numeric_cols:
    if col in df_long.columns:
        df_long[col] = pd.to_numeric(df_long[col], errors="coerce")

#Calculatte mean SBP and BBP + Waist/Hip Ratio
df_long["SBP"] = df_long[["SYSBP1", "SYSBP2", "SYSBP3"]].mean(axis=1, skipna=True)

df_long["DBP"] = df_long[["DIABP1", "DIABP2", "DIABP3"]].mean(axis=1, skipna=True)

df_long["WAIST"] = pd.to_numeric(df_long["WAIST"], errors="coerce")
df_long["HIP"] = pd.to_numeric(df_long["HIP"], errors="coerce")

df_long["WHR"] = df_long["WAIST"] / df_long["HIP"]

In [15]:
df_long[["VISIT",  "SWANID", "SYSBP1", "SYSBP2", "SYSBP3", "SBP", "DIABP1", "DIABP2", "DIABP3", "DBP", "WHR"]].head(20)

,VISIT,SWANID,SYSBP1,SYSBP2,SYSBP3,SBP,DIABP1,DIABP2,DIABP3,DBP,WHR
0,0,10005,114.0,112.0,112.0,112.666667,80.0,80.0,80.0,80.000000,0.709677
1,0,10046,120.0,110.0,116.0,115.333333,58.0,62.0,70.0,63.333333,0.920398
2,0,10056,92.0,90.0,92.0,91.333333,60.0,64.0,66.0,63.333333,0.639394
3,0,10092,108.0,106.0,104.0,106.000000,70.0,68.0,72.0,70.000000,0.816624
4,0,10126,98.0,98.0,102.0,99.333333,72.0,74.0,78.0,74.666667,0.819048
5,0,10153,120.0,122.0,120.0,120.666667,80.0,84.0,84.0,82.666667,0.944276
6,0,10196,82.0,82.0,86.0,83.333333,64.0,62.0,64.0,63.333333,0.832960
7,0,10245,88.0,94.0,90.0,90.666667,62.0,58.0,66.0,62.000000,0.732394
8,0,10258,118.0,114.0,114.0,115.333333,80.0,78.0,78.0,78.666667,0.844828
9,0,10262,120.0,130.0,122.0,124.000000,72.0,80.0,78.0,76.666667,0.777911


In [17]:
df_long["sleep_quality"] = np.where(
    df_long["VISIT"] <= 2,
    df_long["TYPNIGH"].astype("Int64").map({   ##Mapping the TYPNIGH scale to the SLEEPQL scale; Check compare_sleep_variables.ipynb for rationale
        1:1, 
        2:2, 
        3:2, 
        4:3, 
        5:4
    }),
    df_long["SLEEPQL"]
)

In [ ]:
participant_10245 = df_long[
    df_long['SWANID'] == 10245
    ][["VISIT",  "SWANID", "sleep_quality", "TYPNIGH", "SLEEPQL","WAIST","HIP" ]]

print(participant_10245) ##Inspecting the mapping accuracy of a random participant

       VISIT  SWANID  sleep_quality  TYPNIGH  SLEEPQL  WAIST    HIP
7          0   10245            1.0      1.0      NaN   78.0  106.5
3308       1   10245            2.0      3.0      NaN   80.4  107.3
6188       2   10245            2.0      2.0      NaN   85.2  108.4
8936       3   10245            2.0      3.0      2.0   81.1  108.4
11645      4   10245            2.0      NaN      2.0   90.3  112.1
14324      5   10245            2.0      NaN      2.0   90.0  111.5
16941      6   10245            2.0      NaN      2.0   90.0  108.5
19388      7   10245            1.0      NaN      1.0   85.5  109.5
21802      8   10245            NaN      NaN      NaN    NaN    NaN
24080      9   10245            2.0      NaN      2.0   78.4  102.5
26548     10   10245            2.0      NaN      2.0   79.0  100.0


# Inspect random participant trajectory

In [20]:
pid = df_long["SWANID"].iloc[1]

df_long[
df_long["SWANID"] == pid
][[
"VISIT",
"SWANID",
"sleep_quality",
"DHAS",
"SBP",
"WHR"
]].sort_values("VISIT")

,VISIT,SWANID,sleep_quality,DHAS,SBP,WHR
1,0,10046,3.0,342.3,115.333333,0.920398
3302,1,10046,3.0,221.0,109.000000,0.919325
6183,2,10046,3.0,189.7,119.000000,0.881308
8931,3,10046,4.0,185.6,135.000000,0.940734
11640,4,10046,3.0,208.1,98.000000,0.878815
14319,5,10046,4.0,172.7,122.000000,0.929775
16936,6,10046,4.0,145.6,135.000000,0.934529
19384,7,10046,3.0,73.9,125.000000,0.925386
21797,8,10046,3.0,137.0,115.000000,0.919721
24075,9,10046,3.0,183.5,122.000000,0.888158


In [21]:
df_long.to_csv(
    "swan_long_uncensored_clean_v1.csv",
    index=False
)